## Example 1: Basic FastMCP Server Setup

In [ ]:
# Example 1: Basic FastMCP Server Setup
print("FastMCP - Basic Server Setup")

import fastmcp
print(f"✅ FastMCP v{fastmcp.__version__} imported successfully")

# Create FastMCP server instance
server = fastmcp.FastMCP(name="demo-server")

# Define a simple tool
@server.tool()
def greet(name: str) -> str:
    """Greet a person by name"""
    return f"Hello, {name}!"

print(f"✅ Tool 'greet' registered successfully")
print(f"Server name: {server.name}")
print(f"Server type: {type(server).__name__}")


# Call the tool through fastmcp's in-memory client
async with fastmcp.Client(server) as client:
    tools = await client.list_tools()
    result = await client.call_tool("greet", {"name": "Ada"})
assert [t.name for t in tools] == ["greet"], tools
assert result.data == "Hello, Ada!", result
print(f"✅ greet('Ada') via Client -> {result.data}")

print("Example 1 completed")

## Example 2: Tool Implementation and Registration

In [ ]:
# Example 2: Tool Implementation and Registration
print("FastMCP - Tool Implementation")

import fastmcp

# Create server
server = fastmcp.FastMCP(name="tool-server")

# Register multiple tools
@server.tool()
def calculate(operation: str, a: float, b: float) -> float:
    """Perform basic mathematical operations"""
    if operation == "add":
        return a + b
    elif operation == "multiply":
        return a * b
    else:
        raise ValueError(f"Unknown operation: {operation}")

@server.tool()
def text_length(text: str) -> int:
    """Get the length of a text string"""
    return len(text)

print(f"✅ Tools registered successfully")
print(f"Server: {server.name}")


# Call the tools through fastmcp's in-memory client
async with fastmcp.Client(server) as client:
    tool_names = sorted(t.name for t in await client.list_tools())
    added = await client.call_tool("calculate", {"operation": "add", "a": 2, "b": 3})
    multiplied = await client.call_tool("calculate", {"operation": "multiply", "a": 2, "b": 3})
    length = await client.call_tool("text_length", {"text": "fastmcp"})
assert tool_names == ["calculate", "text_length"], tool_names
assert (added.data, multiplied.data, length.data) == (5.0, 6.0, 7), (added.data, multiplied.data, length.data)
print(f"✅ calculate/text_length via Client -> {added.data}, {multiplied.data}, {length.data}")

print("Example 2 completed")

## Example 3: Resource Management

In [ ]:
# Example 3: Resource Management
print("FastMCP - Resource Management")

import fastmcp
import json

server = fastmcp.FastMCP(name="resource-server")

# Register resources
@server.resource("config://settings.json")
def get_config():
    """Get server configuration"""
    return json.dumps({"mode": "production", "debug": False})

@server.resource("data://sample.csv")
def get_sample_data():
    """Get sample dataset"""
    return "id,name,value\n1,item1,100\n2,item2,200"

print(f"✅ Resources registered successfully")
print(f"Resource URIs: config://settings.json, data://sample.csv")
print(f"Server: {server.name}")


# Read the resources back through fastmcp's in-memory client
async with fastmcp.Client(server) as client:
    uris = sorted(str(r.uri) for r in await client.list_resources())
    config_text = (await client.read_resource("config://settings.json"))[0].text
    csv_text = (await client.read_resource("data://sample.csv"))[0].text
assert uris == ["config://settings.json", "data://sample.csv"], uris
assert json.loads(config_text) == {"mode": "production", "debug": False}, config_text
assert csv_text.splitlines() == ["id,name,value", "1,item1,100", "2,item2,200"], csv_text
print(f"✅ Resources read back via Client: {uris}")

print("Example 3 completed")

## Example 4: Prompt Template Management

In [ ]:
# Example 4: Prompt Template Management
print("FastMCP - Prompt Templates")

import fastmcp

server = fastmcp.FastMCP(name="prompt-server")

# Register prompt templates
@server.prompt("system_init")
def system_prompt(context: str = "general"):
    """System initialization prompt"""
    return f"You are an AI assistant with context: {context}. How can I help?"

@server.prompt("task_analysis")
def task_prompt(task: str, priority: str = "medium"):
    """Task analysis prompt"""
    return f"Analyze this task: {task}. Priority: {priority}. Provide a breakdown."

print(f"✅ Prompt templates registered successfully")
print(f"Templates: system_init, task_analysis")
print(f"Server: {server.name}")


# Render the prompts through fastmcp's in-memory client
async with fastmcp.Client(server) as client:
    prompt_names = sorted(p.name for p in await client.list_prompts())
    rendered = await client.get_prompt("task_analysis", {"task": "write docs", "priority": "high"})
    default = await client.get_prompt("system_init", {})
assert prompt_names == ["system_init", "task_analysis"], prompt_names
assert [m.content.text for m in rendered.messages] == ["Analyze this task: write docs. Priority: high. Provide a breakdown."], rendered
assert [m.content.text for m in default.messages] == ["You are an AI assistant with context: general. How can I help?"], default
print(f"✅ Prompt rendered via Client: {rendered.messages[0].content.text}")

print("Example 4 completed")

## Example 5: Complete FastMCP Server with Error Handling

In [ ]:
# Example 5: Complete FastMCP Server with Error Handling
print("FastMCP - Complete Server Implementation")

import fastmcp
import json

# Create comprehensive server
server = fastmcp.FastMCP(name="complete-server")

# Tool with error handling
@server.tool()
def safe_divide(a: float, b: float) -> str:
    """Safely divide two numbers"""
    try:
        if b == 0:
            return "Error: Division by zero"
        return f"Result: {a / b}"
    except Exception as e:
        return f"Error: {str(e)}"

# Resource with JSON data
@server.resource("stats://server")
def server_stats():
    """Get server statistics"""
    return json.dumps({"status": "running", "version": "1.0"})

print(f"✅ Complete server setup successful")
print(f"Tools: safe_divide (with error handling)")
print(f"Resources: stats://server")
print(f"Server: {server.name}")


# Exercise the tool and the resource through fastmcp's in-memory client
async with fastmcp.Client(server) as client:
    divided = await client.call_tool("safe_divide", {"a": 10, "b": 4})
    by_zero = await client.call_tool("safe_divide", {"a": 1, "b": 0})
    stats = json.loads((await client.read_resource("stats://server"))[0].text)
assert divided.data == "Result: 2.5", divided
assert by_zero.data == "Error: Division by zero", by_zero  # the tool's own handled path
assert stats == {"status": "running", "version": "1.0"}, stats
print(f"✅ safe_divide(10, 4) via Client -> {divided.data}")

print("Example 5 completed")